# Шаг 3: дообучение multilingual-e5-base (RU-only и mix-en)

**Что делает ноутбук:** дообучает e5-base контрастивным loss на 100k парах
«запрос → код» в двух конфигурациях и сохраняет модели, логи и метрики на Google Drive.

| эксперимент | пары | anchor | epochs | batch | lr |
|---|---|---|---|---|---|
| `ru-only` | 100k | `query_ru` (перевод NLLB) | 3 | 32 | 1e-5 |
| `mix-en` | 100k (50k RU + 50k EN) | `query_ru` и `query_en` | 3 | 32 | 1e-5 |

У обоих одинаковые loss (`MultipleNegativesRankingLoss`, in-batch негативы), префиксы
e5 (`query: ` / `passage: `) и число шагов (3 × 100k / 32 = 9375) — сравнение честное:
различается только язык anchor'а.

**Почему в Colab:** на локальной машине (16 ГБ RAM, swap занят) системный `earlyoom`
убивает процесс обучения через несколько минут — не хватает памяти, не ошибка в коде
(`research-log.md`, 2026-11-23). Локально готовим данные, тестируем код и оцениваем
baseline; обучение — на GPU.

**Время:** ~2–4 ч на T4 на эксперимент; ноутбук печатает скорость шага, если она
грозит превысить лимит сессии — остановите обучение (чекпоинты сохраняются).

## Что загрузить (Google Drive → папка `semsearch_step3/`)

| файл | откуда | куда |
|---|---|---|
| `train_with_ru.jsonl` | `kursovaya-tex/code/var/data/` | `semsearch_step3/data/` |
| `test_pool.jsonl` | `kursovaya-tex/code/var/data/` | `semsearch_step3/data/` |
| `test_queries_nllb_ru_1000.jsonl` | `kursovaya-tex/code/var/data/` | `semsearch_step3/data/` |
| `manual_queries.jsonl` | `kursovaya-tex/results/` | `semsearch_step3/data/` |

Всё, что ноутбук создаёт (модели, логи, метрики), пишется в `semsearch_step3/out/` и
переживает разрыв сессии.

**Прежде чем запускать:** Runtime → Change runtime type → T4 GPU.

## 1. Проверка GPU

In [ ]:
import torch

assert torch.cuda.is_available(), "GPU нет: Runtime → Change runtime type → T4 GPU"
free_gb, total_gb = [x / 1024**3 for x in torch.cuda.mem_get_info()]
print(f"{torch.cuda.get_device_name(0)}: свободно {free_gb:.1f} из {total_gb:.1f} ГБ; torch {torch.__version__}")

## 2. Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 3. Код и зависимости

Репозиторий клонируется сюда; дальнейшие ячейки работают из
`/content/kry_prompt-lib/kursovaya-tex/code` — там лежит пакет `semsearch`.
Пакеты ставятся из `requirements.txt`, кроме версий, которые в Colab уже стоят вместе
с CUDA-сборкой torch (`torch`) и самого Colab (`numpy`): их замена сломает GPU-сборку.
`numba` нужна `datasets`; версия не фиксирована в `requirements.txt`.

In [ ]:
import os
import subprocess
import sys

if not os.path.exists("/content/kry_prompt-lib"):
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/Krynovv/kry_prompt-lib.git", "/content/kry_prompt-lib"],
        check=True,
    )
CODE = "/content/kry_prompt-lib/kursovaya-tex/code"
os.chdir(CODE)
sys.path.insert(0, CODE)

keep = {"torch", "numpy"}
reqs = [
    line.strip() for line in open("requirements.txt", encoding="utf-8")
    if line.strip() and not line.startswith("#") and line.split("==")[0] not in keep
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *reqs, "numba"], check=True)

import sentence_transformers
import semsearch

print("пакетов из requirements.txt:", len(reqs))
print("sentence-transformers", sentence_transformers.__version__)
print("semsearch:", semsearch.__file__)

## 4. Данные с Drive

In [ ]:
from pathlib import Path

from semsearch.train.finetune import prepare_training_data

ROOT = Path("/content/drive/MyDrive/semsearch_step3")
DATA = ROOT / "data"
OUT = ROOT / "out"
OUT.mkdir(parents=True, exist_ok=True)

files = {
    "train": DATA / "train_with_ru.jsonl",
    "pool": DATA / "test_pool.jsonl",
    "queries": DATA / "test_queries_nllb_ru_1000.jsonl",
    "manual": DATA / "manual_queries.jsonl",
}
missing = [str(path) for path in files.values() if not path.exists()]
assert not missing, f"нет файлов на Drive: {missing} — см. таблицу в начале ноутбука"

for experiment in ("ru-only", "mix-en"):
    print(experiment, "пар:", len(prepare_training_data(files["train"], experiment)))

## 5. Скорость GPU (до полного прогона)

Дообучение шлёт 100k пар по 512 токенов трижды. Один шаг на T4 с batch=32 — 1–3 с;
9375 шагов × 2 c ≈ 5 ч, что может не влезть в бесплатную сессию. Ячейка меряет два
реальных шага (с обратным проходом, тот же loss, что при обучении) и печатает прогноз:
если больше ~4 ч — брать L4/ночную сессию или снизить `epochs`.

In [ ]:
import time

import torch
from sentence_transformers import SentenceTransformer
from sentence_transformers.sentence_transformer import losses

from semsearch.config import EvalConfig
from semsearch.train.finetune import TrainConfig, prepare_training_data, to_dataset

cfg = TrainConfig()
data = to_dataset(prepare_training_data(files["train"], cfg.experiment, limit=64))
batches = [slice(start, start + cfg.batch_size) for start in (0, cfg.batch_size)]

model = SentenceTransformer(EvalConfig().model_name, device="cuda")
model.max_seq_length = cfg.max_seq_length
loss_fn = losses.MultipleNegativesRankingLoss(model)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr)
started = time.time()
for batch in batches:
    sample = [data["anchor"][batch], data["positive"][batch]]
    loss = loss_fn(model, sample)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
torch.cuda.synchronize()
step_seconds = (time.time() - started) / len(batches)
total_steps = cfg.epochs * len(prepare_training_data(files["train"], cfg.experiment)) // cfg.batch_size
print(f"шаг {step_seconds:.2f} с; {total_steps} шагов ≈ {total_steps * step_seconds / 3600:.1f} ч")
del model, optimizer, loss_fn
torch.cuda.empty_cache()

## 6. Обучение

`train_finetune(config, train, queries, pool, manual)` обучает и сохраняет модель в
`out/<эксперимент>/` (`train_config.json` рядом). Лог `out/train_log_<эксперимент>.csv`:

- строка `step=0` — метрики до обучения (должны совпасть с baseline Шага 2: EN 0.7575 /
  RU 0.5256 MRR@10);
- дальше каждые `EVAL_EVERY` шагов — loss и MRR@10/Recall@10 на EN и RU по тест-корпусу
  5000 функций и ручным запросам;
- последняя строка — после последней эпохи.

Итоговые метрики этой же ячейкой пишутся в `out/metrics_<эксперимент>.json`.

Чекпоинты каждые 1000 шагов лежат в `out/<эксперимент>/checkpoints` (последние два,
~1.1 ГБ). Если сессия оборвалась, перезапустите ячейку: Trainer сам подхватит последний
чекпоинт, конфиг не меняйте.

Сначала идёт `mix-en` (он чувствительнее к забыванию английского), потом `ru-only`; чтобы
оставить один — уберите второй из списка.


In [ ]:
import json
import time

from semsearch.train.finetune import TrainConfig, train_finetune

EXPERIMENTS = ["mix-en", "ru-only"]
EVAL_EVERY = 500

for experiment in EXPERIMENTS:
    started = time.time()
    print(f"=== {experiment} ===", flush=True)
    save_dir = OUT / experiment
    metrics = train_finetune(
        TrainConfig(experiment=experiment, save_dir=save_dir, eval_every=EVAL_EVERY),
        files["train"],
        files["queries"],
        files["pool"],
        files["manual"],
    )
    (OUT / f"metrics_{experiment}.json").write_text(
        json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print(f"{experiment}: {(time.time() - started) / 3600:.1f} ч, итог {metrics}; "
          f"модель {save_dir}", flush=True)


## 7. Что забрать с Drive

`semsearch_step3/out/`:

- `train_log_ru-only.csv`, `train_log_mix-en.csv` — loss и метрики по шагам (в т.ч. RU по
  ручным запросам автора);
- `metrics_ru-only.json`, `metrics_mix-en.json` — итоговые метрики;
- `ru-only/`, `mix-en/` — модели и `train_config.json`.

Модели из Colab проверяются локально тем же путём, что и baseline Шага 2:
`python -m semsearch.train_cli evaluate --model <путь> --queries ... --pool ...`.
